In [ ]:
%%sql -r dataframe_1
SELECT CURRENT_VERSION();

In [ ]:
%%sql -r dataframe_2
CREATE WAREHOUSE SALES_WH
WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

In [ ]:
%%sql -r dataframe_3
SHOW WAREHOUSES LIKE 'SALES_WH';

In [ ]:
%%sql -r dataframe_4
USE WAREHOUSE SALES_WH;

In [ ]:
%%sql -r dataframe_5
SELECT CURRENT_WAREHOUSE();

In [ ]:
%%sql -r dataframe_6
CREATE DATABASE CUSTOMER_SALES_DB;

In [ ]:
%%sql -r dataframe_7
SHOW DATABASES LIKE 'CUSTOMER_SALES_DB';

In [ ]:
%%sql -r dataframe_8
USE DATABASE CUSTOMER_SALES_DB;

In [ ]:
%%sql -r dataframe_9
SELECT CURRENT_DATABASE();

In [ ]:
%%sql -r dataframe_10
CREATE SCHEMA SALES_SCHEMA;

In [ ]:
%%sql -r dataframe_11
SHOW SCHEMAS LIKE 'SALES_SCHEMA';

In [ ]:
%%sql -r dataframe_12
USE SCHEMA SALES_SCHEMA;

In [ ]:
%%sql -r dataframe_13
SELECT CURRENT_SCHEMA();

In [ ]:
%%sql -r dataframe_14
SELECT
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
%%sql -r dataframe_15
CREATE FILE FORMAT CSV_FILE_FORMAT
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_DELIMITER = ',';

In [ ]:
%%sql -r dataframe_16
SHOW FILE FORMATS LIKE 'CSV_FILE_FORMAT';

In [ ]:
%%sql -r dataframe_17
CREATE STAGE SALES_STAGE
    FILE_FORMAT = CSV_FILE_FORMAT;

In [ ]:
%%sql -r dataframe_18
SHOW STAGES LIKE 'SALES_STAGE';

In [ ]:
%%sql -r dataframe_19
LIST @SALES_STAGE;

In [ ]:
%%sql -r dataframe_20
CREATE TABLE CUSTOMERS (
    customer_id NUMBER(38,0) NOT NULL,
    first_name  VARCHAR(100) NOT NULL,
    last_name   VARCHAR(100) NOT NULL,
    email       VARCHAR(255) NOT NULL,
    phone       VARCHAR(20),
    address     VARCHAR(255)
);

In [ ]:
%%sql -r dataframe_21
DESCRIBE TABLE CUSTOMERS;

In [ ]:
%%sql -r dataframe_22
CREATE TABLE FOODITEMS (
    food_id       NUMBER(38,0) NOT NULL,
    name          VARCHAR(200) NOT NULL,
    price         NUMBER(10,2) NOT NULL,
    category      VARCHAR(100) NOT NULL,
    availability  VARCHAR(20) NOT NULL
);

In [ ]:
%%sql -r dataframe_23
DESCRIBE TABLE FOODITEMS;

In [ ]:
%%sql -r dataframe_24
CREATE TABLE ORDERS (
    order_id      NUMBER(38,0) NOT NULL,
    customer_id   NUMBER(38,0) NOT NULL,
    food_id       NUMBER(38,0) NOT NULL,
    quantity      NUMBER(10,0) NOT NULL,
    order_date    TIMESTAMP_NTZ NOT NULL,
    status        VARCHAR(30) NOT NULL,
    total_amount  NUMBER(12,2) NOT NULL
);

In [ ]:
%%sql -r dataframe_25
DESCRIBE TABLE ORDERS;

In [ ]:
%%sql -r dataframe_26
COPY INTO CUSTOMERS
FROM @SALES_STAGE/customers.csv
FILE_FORMAT = (
    FORMAT_NAME = 'CSV_FILE_FORMAT'
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_27
COPY INTO FOODITEMS
FROM @SALES_STAGE/fooditems.csv
FILE_FORMAT = (
    FORMAT_NAME = 'CSV_FILE_FORMAT'
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_28
COPY INTO ORDERS
FROM @SALES_STAGE/orders.csv
FILE_FORMAT = (
    FORMAT_NAME = 'CSV_FILE_FORMAT'
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_29
SELECT 'CUSTOMERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM CUSTOMERS

UNION ALL

SELECT 'FOODITEMS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM FOODITEMS

UNION ALL

SELECT 'ORDERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM ORDERS;

In [ ]:
%%sql -r dataframe_30
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    SUM(o.total_amount) AS total_spent
FROM CUSTOMERS c
JOIN ORDERS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
ORDER BY total_spent DESC;

In [ ]:
%%sql -r dataframe_31
SELECT
    SUM(total_amount) AS total_revenue
FROM ORDERS;

In [ ]:
%%sql -r dataframe_32
SELECT
    f.category,
    SUM(o.total_amount) AS category_revenue
FROM FOODITEMS f
JOIN ORDERS o
    ON f.food_id = o.food_id
GROUP BY f.category
ORDER BY category_revenue DESC;

In [ ]:
%%sql -r dataframe_33
SELECT
    status,
    SUM(total_amount) AS status_revenue
FROM ORDERS
GROUP BY status
ORDER BY status_revenue DESC;

In [ ]:
%%sql -r dataframe_34
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    COUNT(o.order_id) AS order_count
FROM CUSTOMERS c
JOIN ORDERS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
ORDER BY order_count DESC;

In [ ]:
%%sql -r dataframe_35
SELECT
    order_id,
    customer_id,
    food_id,
    quantity,
    order_date,
    status,
    total_amount
FROM ORDERS
WHERE status = 'Delivered'
ORDER BY order_id;

In [ ]:
%%sql -r dataframe_36
SELECT
    o.order_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    o.order_date,
    o.status,
    o.total_amount
FROM ORDERS o
JOIN CUSTOMERS c
    ON o.customer_id = c.customer_id
WHERE o.order_date > '2026-07-12'
ORDER BY o.order_date;

In [ ]:
%%sql -r dataframe_37
CREATE OR REPLACE VIEW CUSTOMER_SALES_REPORT AS
SELECT
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    SUM(o.total_amount) AS total_amount_spent
FROM CUSTOMERS c
JOIN ORDERS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name;

In [ ]:
%%sql -r dataframe_38
SELECT *
FROM CUSTOMER_SALES_REPORT;

In [ ]:
%%sql -r dataframe_39
SELECT *
FROM CUSTOMER_SALES_REPORT
ORDER BY total_amount_spent DESC;